# SepsiScope: exploratory data analysis (Phase 1)

PhysioNet/CinC Challenge 2019 training sets A and B, after cleaning. This notebook only **calls functions from `sepsiscope.data`**; all logic and tests live in the package. No features are engineered and no model is trained here.

Run `uv run sepsiscope data download`, `data verify` and `data build` first. Outputs are stripped before commit (nbstripout); the findings are written up in `docs/eda_findings.md`.

In [ ]:
import pandas as pd

from sepsiscope.config import REPO_ROOT, load_config
from sepsiscope.data.pipeline import run_eda

cfg = load_config()
tables = run_eda(
    REPO_ROOT / cfg.paths.data_dir,
    figures_dir=REPO_ROOT / cfg.paths.reports_dir / "figures",
)
pd.set_option("display.float_format", "{:,.3f}".format)
sorted(tables)

## 1. Cohort by hospital

In [ ]:
tables["cohort"]

## 2. Missingness per variable, by hospital

Share of rows (patient-hours) where the variable is missing. Figure: `reports/figures/eda_missingness.png`.

In [ ]:
tables["missingness"].sort_values("A", ascending=False)

## 3. Measurement frequency

Measurements per patient-day, and the share of patients who ever have the variable measured. Figure: `reports/figures/eda_measurement_frequency.png`.

In [ ]:
tables["frequency"].sort_values(("per_patient_day", "A"), ascending=False)

## 4. Stay length, septic vs not

Figure: `reports/figures/eda_stay_length.png`.

In [ ]:
patients = tables["patients"]
patients.groupby(["hospital", "septic"])["rows"].describe()

## 5. Label timing

ICU hour (`ICULOS`) at which the label first turns positive, and how many rows remain from that point. Figure: `reports/figures/eda_label_timing.png`.

In [ ]:
septic = patients[patients["septic"]]
timing = ["first_positive_iculos", "first_positive_t", "rows_from_first_positive"]
septic.groupby("hospital")[timing].describe().T

## 6. Distribution shift between hospitals

Jensen-Shannon divergence (base 2) of the observed values, A vs B, on shared equal-count bins, plus the gap in missingness. Figure: `reports/figures/eda_hospital_shift.png`.

In [ ]:
tables["shift"].sort_values("js_values", ascending=False)

## 7. Unit1 / Unit2 as a hospital fingerprint

Figure: `reports/figures/eda_unit_fingerprint.png`. The last number is how often a single counting rule on the unit columns guesses a patient's hospital correctly.

In [ ]:
display(tables["units"])
tables["unit_rule_accuracy"]